# Uzbek Low-Resource RAG Cookbook

This cookbook demonstrates how to build a culturally grounded RAG retrieval workflow for **Uzbek**, a low-resource Central Asian language, using the public SOAS English-Uzbek RAG evaluation benchmark as the evaluation harness.

**Headline result:** Targeted Uzbek source curation improved retrieval recall from **39% to 98%** (Cohen's *d* = 2.91) in the source benchmark.

**Source dataset:**
- Hugging Face: [`rajantripathi/soas-english-uzbek-rag-evaluation`](https://huggingface.co/datasets/rajantripathi/soas-english-uzbek-rag-evaluation)
- GitHub: https://github.com/rajantripathi/soas-rag-evaluation
- License: CC-BY-4.0 (dataset), MIT (code)


## Setup

```bash
pip install langchain langchain-community langchain-chroma sentence-transformers datasets
```


In [ ]:
from datasets import load_dataset

DATASET_ID = 'rajantripathi/soas-english-uzbek-rag-evaluation'
RAW_JSONL = 'https://raw.githubusercontent.com/rajantripathi/soas-rag-evaluation/main/hf_dataset/manual_eval_v5_retrieval_only.jsonl'

try:
    ds = load_dataset(DATASET_ID, split='train')
except Exception:
    # Public GitHub fallback for environments where the HF mirror is unavailable.
    ds = load_dataset('json', data_files=RAW_JSONL, split='train')

print(f'Loaded {len(ds)} rows. Languages:', set(ds['language']))
print('Sample:', ds[0])

## Build a Chroma vector store over Uzbek source records

We use `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, a small multilingual encoder that supports Uzbek and runs on CPU. In a production RAG system, replace the stand-in documents below with the actual Uzbek source passages.


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

uz_rows = [r for r in ds if r['language'] == 'uz']
documents = [
    Document(
        page_content=r.get('source_title') or r['question'],
        metadata={
            'source_doc_id': r['source_doc_ids'][0] if r.get('source_doc_ids') else None,
            'id': r['id'],
            'domain': r['domain'],
        },
    )
    for r in uz_rows
]
vectorstore = Chroma.from_documents(documents, embeddings, collection_name='uz_cookbook')
print(f'Indexed {len(documents)} documents.')

## Evaluate retrieval recall@5

An item is counted as retrieved when any of the top-5 returned document IDs matches `source_doc_ids`.


In [ ]:
def retrieval_recall_at_k(retrieved, source_doc_ids, k=5):
    return int(bool(set(retrieved[:k]) & set(source_doc_ids or [])))

hits = 0
for row in uz_rows:
    docs = vectorstore.similarity_search(row['question'], k=5)
    retrieved_ids = [doc.metadata.get('source_doc_id') for doc in docs]
    hits += retrieval_recall_at_k(retrieved_ids, row['source_doc_ids'], k=5)

recall = hits / len(uz_rows)
print(f'Uzbek retrieval recall@5 (cookbook demo): {recall:.1%}')
print('Note: In the published benchmark, the full corpus raises this from 39% to 98%.')

## Why corpus coverage dominates model choice

On this evaluation set:

| Setting | Retrieval Recall |
| --- | ---: |
| English baseline | 63% |
| Uzbek baseline (no targeted corpus) | 39% |
| Uzbek after corpus supplementation | **98%** |

**Interpretation:** For low-resource languages, source corpus coverage can be a first-order bottleneck. Before swapping embedding models or LLMs, audit whether the retriever has access to the documents that actually contain the answer.

Full methodology, statistical analysis, and reproducibility notes are in the source repository: https://github.com/rajantripathi/soas-rag-evaluation


## Citation

```bibtex
@dataset{tripathi_2026_soas_en_uz_rag,
  author       = {Tripathi, Rajan Prasad},
  title        = {SOAS English-Uzbek RAG Evaluation (Retrieval-Only)},
  year         = {2026},
  publisher    = {Hugging Face},
  version      = {manual_eval_v5},
  url          = {https://huggingface.co/datasets/rajantripathi/soas-english-uzbek-rag-evaluation}
}
```
